In [1]:
# ============================================================
# Transfer Learning using ResNet on CIFAR-10
# ============================================================
# Description  : Pretend CIFAR-10 categories are Bengali
#                celebrities — same transfer learning concept
# Model        : ResNet18 pretrained on ImageNet
# Author       : Fatima
# ============================================================

# ── Import Required Libraries ────────────────────────────────
import torch                                 # PyTorch
import torch.nn as nn                        # neural network
import torchvision.models as models          # pretrained models
import torchvision.transforms as transforms  # image transforms
from torchvision.datasets import CIFAR10     # practice dataset
from torch.utils.data import DataLoader      # data feeding
import matplotlib.pyplot as plt              # visualization

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# ── Step 2: Load Pretrained ResNet18 ────────────────────────
# ResNet18 = pretrained on ImageNet (1000 categories)
# already knows edges, shapes, textures, face patterns
# we borrow this knowledge for our celebrity task

print("Loading pretrained ResNet18...")

# weights=DEFAULT loads the pretrained ImageNet weights
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# look at the final layer before we change it
print("\nOriginal final layer:")
print(model.fc)

# ── Step 3: Replace Final Layer ─────────────────────────────
# original final layer → Linear(512, 1000) = 1000 categories
# we replace it       → Linear(512, 10)   = 10 celebrities
# (in real project    → Linear(512, 250)  = 250 Bengali celebs)

num_celebrities = 10  # pretending 10 CIFAR categories = 10 celebrities

model.fc = nn.Linear(512, num_celebrities)

print("\nModified final layer:")
print(model.fc)

print("\nResNet18 ready for celebrity recognition! ✅")

Loading pretrained ResNet18...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 344MB/s]


Original final layer:
Linear(in_features=512, out_features=1000, bias=True)

Modified final layer:
Linear(in_features=512, out_features=10, bias=True)

ResNet18 ready for celebrity recognition! ✅


In [3]:
# ── Step 3: Load CIFAR-10 Dataset ───────────────────────────
# CIFAR-10 = 60,000 images, 10 categories
# we pretend each category = one Bengali celebrity
# ResNet expects images of size 224x224
# so we resize CIFAR-10 images from 32x32 to 224x224

print("Loading CIFAR-10 dataset...")

# define image transformations
transform = transforms.Compose([
    # resize to 224x224 (ResNet expected input size)
    transforms.Resize((224, 224)),
    # convert image to PyTorch tensor
    transforms.ToTensor(),
    # normalize using ImageNet mean and std
    # because ResNet was trained with these values
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# load training data
train_dataset = CIFAR10(
    root="./data",      # where to save dataset
    train=True,         # training set
    download=True,      # download if not present
    transform=transform # apply transformations
)

# load test data
test_dataset = CIFAR10(
    root="./data",
    train=False,        # test set
    download=True,
    transform=transform
)

# create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,      # 32 images at a time
    shuffle=True        # randomize each epoch
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False       # no need to shuffle test data
)

print(f"Training images : {len(train_dataset)}")
print(f"Test images     : {len(test_dataset)}")
print(f"Training batches: {len(train_loader)}")
print("Dataset ready! ✅")

Loading CIFAR-10 dataset...


100%|██████████| 170M/170M [41:04<00:00, 69.2kB/s]


Training images : 50000
Test images     : 10000
Training batches: 1563
Dataset ready! ✅


In [7]:
# ── Step 4: Training Loop (Fast Version) ────────────────────
# use only 500 images instead of 50,000
# so it runs fast on CPU
# concept is exactly the same!

from torch.utils.data import Subset

# take only first 500 training images
small_train = Subset(train_dataset, range(500))
small_loader = DataLoader(small_train,
                          batch_size=32,
                          shuffle=True)

# setup
device = torch.device("cpu")
print(f"Using device: {device}")
model = model.to(device)

# loss and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Training on 500 images ({len(small_loader)} batches)")
print("Starting training...\n")

for epoch in range(2):

    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(small_loader):

        images = images.to(device)
        labels = labels.to(device)

        # Step 1: forward pass
        predictions = model(images)

        # Step 2: calculate loss
        loss = loss_fn(predictions, labels)

        # Step 3: backpropagation
        loss.backward()

        # Step 4: update weights
        optimizer.step()

        # Step 5: reset gradients
        optimizer.zero_grad()

        total_loss += loss.item()
        _, predicted = torch.max(predictions, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        print(f"Epoch {epoch+1} | Batch {batch_idx+1}/{len(small_loader)} | Loss: {round(loss.item(), 4)}")

    accuracy = round((correct / total) * 100, 2)
    avg_loss = round(total_loss / len(small_loader), 4)
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Average Loss : {avg_loss}")
    print(f"  Accuracy     : {accuracy}%\n")

print("Training complete! ✅")

Using device: cpu
Training on 500 images (16 batches)
Starting training...

Epoch 1 | Batch 1/16 | Loss: 0.7116
Epoch 1 | Batch 2/16 | Loss: 1.3489
Epoch 1 | Batch 3/16 | Loss: 1.1916
Epoch 1 | Batch 4/16 | Loss: 1.2354
Epoch 1 | Batch 5/16 | Loss: 1.4737
Epoch 1 | Batch 6/16 | Loss: 1.1769
Epoch 1 | Batch 7/16 | Loss: 1.4508
Epoch 1 | Batch 8/16 | Loss: 1.3606
Epoch 1 | Batch 9/16 | Loss: 1.7716
Epoch 1 | Batch 10/16 | Loss: 1.4312
Epoch 1 | Batch 11/16 | Loss: 1.4026
Epoch 1 | Batch 12/16 | Loss: 0.9197
Epoch 1 | Batch 13/16 | Loss: 2.539
Epoch 1 | Batch 14/16 | Loss: 1.0783
Epoch 1 | Batch 15/16 | Loss: 1.2675
Epoch 1 | Batch 16/16 | Loss: 1.628

Epoch 1 Summary:
  Average Loss : 1.3742
  Accuracy     : 57.4%

Epoch 2 | Batch 1/16 | Loss: 0.9406
Epoch 2 | Batch 2/16 | Loss: 0.6462
Epoch 2 | Batch 3/16 | Loss: 0.4644
Epoch 2 | Batch 4/16 | Loss: 0.9182
Epoch 2 | Batch 5/16 | Loss: 0.9893
Epoch 2 | Batch 6/16 | Loss: 0.7247
Epoch 2 | Batch 7/16 | Loss: 0.8304
Epoch 2 | Batch 8/16 | Lo

In [8]:
# ── Step 5: Evaluate on Test Data ───────────────────────────
# test the model on images it has NEVER seen before
# this tells us if model truly learned or just memorized

from torch.utils.data import Subset

# take 200 test images
small_test = Subset(test_dataset, range(200))
small_test_loader = DataLoader(small_test,
                               batch_size=32,
                               shuffle=False)

# switch model to evaluation mode
# turns off dropout and batch normalization
model.eval()

correct = 0
total = 0

# torch.no_grad() = don't calculate gradients
# we're just testing, not training
with torch.no_grad():
    for images, labels in small_test_loader:

        images = images.to(device)
        labels = labels.to(device)

        # forward pass only
        predictions = model(images)

        # get predicted class
        _, predicted = torch.max(predictions, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

accuracy = round((correct / total) * 100, 2)
print(f"Test Accuracy: {accuracy}%")
print(f"Correct: {correct}/{total}")

Test Accuracy: 52.0%
Correct: 104/200
